# Lab Exercise 10: Learning the XOR Boolean Function Using an MLP

**Name:** SOvin Somy
**Reg. No.:** 2547157
**Course:** MCA – Deep Learning Lab

**Aim:** To implement an MLP that learns the XOR function using Keras, PyTorch, and low-level TensorFlow, and compare the three.


## Dataset

In [1]:
import numpy as np

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=np.float32)
y = np.array([[0], [1], [1], [0]], dtype=np.float32)


## 1. Keras Implementation

2 inputs → 4 hidden neurons (Tanh) → 1 output (Sigmoid). Used Tanh instead of ReLU because ReLU kept getting stuck at 75% accuracy on this tiny dataset (dead neuron problem).

In [2]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(4, activation='tanh'),
    layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.05),
              loss='binary_crossentropy', metrics=['accuracy'])
history = model.fit(X, y, epochs=500, verbose=0)

print(f"Final loss: {history.history['loss'][-1]:.4f}, Accuracy: {history.history['accuracy'][-1]:.4f}")


Final loss: 0.0013, Accuracy: 1.0000


In [3]:
preds = (model.predict(X, verbose=0) > 0.5).astype(int)
for i in range(4):
    print(f"Input: {X[i]}  Predicted: {preds[i][0]}  Actual: {int(y[i][0])}")


Input: [0. 0.]  Predicted: 0  Actual: 0
Input: [0. 1.]  Predicted: 1  Actual: 1
Input: [1. 0.]  Predicted: 1  Actual: 1
Input: [1. 1.]  Predicted: 0  Actual: 0


## 2. PyTorch Implementation

Same architecture, but with an explicit training loop (forward pass → loss → backward → optimizer step). Used Tanh here too, for the same reason as the Keras model — ReLU occasionally got stuck at 75% accuracy on this 4-point dataset.

In [4]:
import torch
import torch.nn as nn

torch.manual_seed(0)

X_t = torch.tensor(X)
y_t = torch.tensor(y)

class XORNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 4)
        self.output = nn.Linear(4, 1)
    def forward(self, x):
        x = torch.tanh(self.hidden(x))
        return torch.sigmoid(self.output(x))

torch_model = XORNet()
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(torch_model.parameters(), lr=0.05)

for epoch in range(500):
    optimizer.zero_grad()
    out = torch_model(X_t)
    loss = criterion(out, y_t)
    loss.backward()
    optimizer.step()

acc = ((out > 0.5).float() == y_t).float().mean().item()
print(f"Final loss: {loss.item():.4f}, Accuracy: {acc:.4f}")


Final loss: 0.0005, Accuracy: 1.0000


In [5]:
with torch.no_grad():
    preds = (torch_model(X_t) > 0.5).int()
for i in range(4):
    print(f"Input: {X[i]}  Predicted: {preds[i][0].item()}  Actual: {int(y[i][0])}")


Input: [0. 0.]  Predicted: 0  Actual: 0
Input: [0. 1.]  Predicted: 1  Actual: 1
Input: [1. 0.]  Predicted: 1  Actual: 1
Input: [1. 1.]  Predicted: 0  Actual: 0


## 3. TensorFlow Low-Level Implementation

No Keras layers — weights and biases as `tf.Variable`, gradients computed manually with `tf.GradientTape`, and weight updates applied by hand (`w := w - lr * grad`). This is what `.fit()`/`optimizer.step()` are doing internally.

In [6]:
import tensorflow as tf

W1 = tf.Variable(tf.random.normal([2, 4], stddev=0.5))
b1 = tf.Variable(tf.zeros([4]))
W2 = tf.Variable(tf.random.normal([4, 1], stddev=0.5))
b2 = tf.Variable(tf.zeros([1]))

X_tf, y_tf = tf.constant(X), tf.constant(y)

def forward(x):
    h = tf.nn.relu(tf.matmul(x, W1) + b1)
    return tf.nn.sigmoid(tf.matmul(h, W2) + b2)

def bce_loss(y_true, y_pred):
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
    return -tf.reduce_mean(y_true * tf.math.log(y_pred) + (1 - y_true) * tf.math.log(1 - y_pred))

lr = 0.1
for epoch in range(500):
    with tf.GradientTape() as tape:
        preds = forward(X_tf)
        loss = bce_loss(y_tf, preds)
    grads = tape.gradient(loss, [W1, b1, W2, b2])
    for var, g in zip([W1, b1, W2, b2], grads):
        var.assign_sub(lr * g)

acc = tf.reduce_mean(tf.cast(tf.equal(tf.cast(preds > 0.5, tf.float32), y_tf), tf.float32)).numpy()
print(f"Final loss: {loss.numpy():.4f}, Accuracy: {acc:.4f}")


Final loss: 0.2822, Accuracy: 1.0000


In [7]:
final_preds = (forward(X_tf).numpy() > 0.5).astype(int)
for i in range(4):
    print(f"Input: {X[i]}  Predicted: {final_preds[i][0]}  Actual: {int(y[i][0])}")


Input: [0. 0.]  Predicted: 0  Actual: 0
Input: [0. 1.]  Predicted: 1  Actual: 1
Input: [1. 0.]  Predicted: 1  Actual: 1
Input: [1. 1.]  Predicted: 0  Actual: 0


## Conclusion

All three implementations learned the XOR function correctly, reaching close to 100% accuracy. This confirms that a single-layer perceptron cannot solve XOR since it isn't linearly separable, but adding a hidden layer with a non-linear activation lets the network form a curved decision boundary that separates the classes correctly. The three libraries gave the same result because they implement the same underlying math (forward pass, BCE loss, backpropagation, gradient descent) — they just differ in how much of that process is automated for the programmer. One practical thing I noticed while running this: my first Keras attempt used ReLU in the hidden layer and got stuck at 75% accuracy instead of 100%. Switching to Tanh fixed it, since ReLU can occasionally get "stuck" at zero output for a very small dataset like this. Keras was the fastest to write, PyTorch made the training loop explicit, and the low-level TensorFlow version was the most useful for actually understanding what backpropagation and gradient descent are doing step by step.
